### **Basic LCEL**

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

model = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0.0
)

# Step 1 — Define each piece
template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer in 2 lines max."),
    ("human", "{question}")
])

parser = StrOutputParser()

# Step 2 — Connect with pipe operator
chain = template | model | parser

# Step 3 — Invoke the whole chain with ONE call
result = chain.invoke({"question": "What is machine learning?"})

print(result)  # clean string, no .content needed

Machine learning is a subset of artificial intelligence that enables computers to learn from data and improve their performance on a specific task without being explicitly programmed. It involves training algorithms to recognize patterns, make predictions, and classify data based on historical information.


### **Chain Supports Stream Automatically**

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

model = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0.7
)

template = ChatPromptTemplate.from_messages([
    ("system", "You are a storyteller."),
    ("human", "Tell me a short story about {topic}")
])

parser = StrOutputParser()

chain = template | model | parser

# Stream the chain — word by word output
for chunk in chain.stream({"topic": "a robot learning to code"}):
    print(chunk, end="", flush=True)

### **Parallel Chaining**

In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

load_dotenv()

model = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0.7
)

# Chain 1 — summarize
summarize_template = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following text in one line."),
    ("human", "{text}")
])

# Chain 2 — sentiment
sentiment_template = ChatPromptTemplate.from_messages([
    ("system", "Analyze sentiment. Reply with just: Positive, Negative, or Neutral."),
    ("human", "{text}")
])

# Chain 3 — keywords
keyword_template = ChatPromptTemplate.from_messages([
    ("system", "Extract 3 keywords from the text. Reply with just the keywords separated by commas."),
    ("human", "{text}")
])

parser = StrOutputParser()

# Build individual chains
summarize_chain = summarize_template | model | parser
sentiment_chain = sentiment_template | model | parser
keyword_chain = keyword_template | model | parser

# Run ALL THREE in parallel
parallel_chain = RunnableParallel(
    summary=summarize_chain,
    sentiment=sentiment_chain,
    keywords=keyword_chain
)

# One invoke — three results at same time
result = parallel_chain.invoke({
    "text": """
    LangChain is an amazing framework for building AI applications.
    It makes working with language models so much easier and faster.
    I absolutely love using it for my projects!
    """
})

print("Summary:", result["summary"])
print("Sentiment:", result["sentiment"])
print("Keywords:", result["keywords"])

Summary: LangChain is a framework that simplifies and accelerates the development of AI applications by making it easier to work with language models.
Sentiment: Positive
Keywords: LangChain, AI, applications
